### Testing GraphBuilder

#### Node classification - One Graph - Cora

In [2]:
from torch_geometric import datasets
from torch_geometric.transforms import GDC
from rewiring.model.GCN import GCN
from rewiring.pipeline.SingleGraphNodeClassificationPipeline import SingleGraphNodeClassificationPipeline

from rewiring.transform.AddRandomEdges import AddRandomEdges


import torch
import numpy as np
import logging

# logging.basicConfig(level=logging.INFO)

# transform = AddRandomEdges(min_density=0.2)
# transform = None
transform = GDC()
dset = datasets.Planetoid(root=f'./../res/datasets/citeseer', name='CiteSeer', transform=transform)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GCN(num_classes=dset.num_classes, 
            num_features=dset.num_features, 
            hidden_size=16)

model.set_classification_task(task='node')
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)
criterion = torch.nn.NLLLoss()

pipeline = SingleGraphNodeClassificationPipeline(
    device=device,
    model=model,
    criterion=criterion,
    optimizer=optimizer
)

pipeline.set_masks(data=dset)

_, _, acc = pipeline.train_validate(
    data=dset[0],
    epochs=200,
    log=True
)
np.array(acc).max()

Epoch: [1/200]         Training loss: 1.8028         Validation loss: 1.7880         Accuracy: 0.1922
Epoch: [2/200]         Training loss: 1.7804         Validation loss: 1.7619         Accuracy: 0.3303
Epoch: [3/200]         Training loss: 1.7534         Validation loss: 1.7334         Accuracy: 0.3348
Epoch: [4/200]         Training loss: 1.7232         Validation loss: 1.7006         Accuracy: 0.3318
Epoch: [5/200]         Training loss: 1.6883         Validation loss: 1.6641         Accuracy: 0.3168
Epoch: [6/200]         Training loss: 1.6491         Validation loss: 1.6290         Accuracy: 0.3078
Epoch: [7/200]         Training loss: 1.6104         Validation loss: 1.5944         Accuracy: 0.3153
Epoch: [8/200]         Training loss: 1.5720         Validation loss: 1.5580         Accuracy: 0.3183
Epoch: [9/200]         Training loss: 1.5323         Validation loss: 1.5190         Accuracy: 0.3498
Epoch: [10/200]         Training loss: 1.4907         Validation loss: 1.4798     

0.7522522522522522

#### Node Classification - Multiple Graphs - LRGB PascalVOC-SP

In [14]:
import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch_geometric.nn.conv import GCNConv
from torch_geometric.loader import DataLoader
from torch_geometric.datasets import LRGBDataset

from rewiring.model.GCN import GCN
from rewiring.transform.AddRandomEdges import AddRandomEdges
from rewiring.transform.ReduceDiameter import ReduceDiameter
from rewiring.pipeline.NodeClassificationPipeline import NodeClassifiactionPipeline

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

batch_size = 64
# transform = ReduceDiameter(max_diam=200)
# transform = AddRandomEdges(min_density=0.25)
transform = None

# Load the dataset
print("Loading and transforming training dataset...")
train_dataset = LRGBDataset(root=f'./../res/datasets/LRGB', name='PascalVOC-SP', transform=transform)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

val_dataset = LRGBDataset(root=f'./../res/datasets/LRGB', name='PascalVOC-SP', split='val')
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

test_dataset = LRGBDataset(root=f'./../res/datasets/LRGB', name='PascalVOC-SP', split='test')
test_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# Create the model
num_features = train_dataset.num_features
num_classes = train_dataset.num_classes
hidden_size = 64
model = GCN(num_classes=num_classes,
            num_features=num_features,
            hidden_size=hidden_size).to(device)
model.set_classification_task('node')

# Set the loss function and optimizer
criterion = nn.NLLLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

pipeline = NodeClassifiactionPipeline(
    device = device,
    model=model,
    criterion=criterion,
    optimizer=optimizer
)

num_epochs = 10
print("Started training...")
pipeline.train_validate(
    train_data=train_loader, 
    val_data=val_loader, 
    epochs=num_epochs, 
    log=True
)
print("Training finished!")

print("Starting testing...")
model.eval()
correct = 0
epoch_test_loss = []
sum = len(test_loader.dataset.y)
print(f'Test element count: {sum}')
for batch in test_loader:
    batch = batch.to(device)
    with torch.no_grad():
        out = model(batch.x, batch.edge_index)
        test_loss = criterion(out, batch.y)
        epoch_test_loss.append(test_loss.item())
        _, pred = torch.max(out.data, 1)
        correct += (pred == batch.y).float().sum().item()

acc = correct / sum
test_loss = np.array(epoch_test_loss).mean().item()

print(f"""Loss: {test_loss:.4f} \
        Accuracy: {acc:.4f}""")

Loading and transforming training dataset...


Extracting ../res/datasets/LRGB/pascalvocsp.zip
Processing...
Processing test dataset: 100%|██████████| 1429/1429 [00:00<00:00, 38779.11it/s]
Done!


Started training...
Epoch: [1/10]         Training loss: 3.0466         Validation loss: 1.6044         Accuracy: 0.6964
Epoch: [2/10]         Training loss: 1.5615         Validation loss: 1.5360         Accuracy: 0.6964
Epoch: [3/10]         Training loss: 1.5118         Validation loss: 1.4882         Accuracy: 0.6964
Epoch: [4/10]         Training loss: 1.4746         Validation loss: 1.4751         Accuracy: 0.6932
Epoch: [5/10]         Training loss: 1.4611         Validation loss: 1.4466         Accuracy: 0.6964
Epoch: [6/10]         Training loss: 1.4435         Validation loss: 1.4435         Accuracy: 0.6964


KeyboardInterrupt: 

#### Graph Classification - Proteins, Mutag

In [10]:
from torch_geometric import datasets
from torch_geometric.loader import DataLoader
from torch_geometric.transforms import TwoHop
from rewiring.model.GCN import GCN
from rewiring.TrainValTestSplitter import TrainValTestSplitter
from rewiring.transform.ReduceDiameter import ReduceDiameter
from rewiring.transform.AddRandomEdges import AddRandomEdges
from rewiring.pipeline.GraphClassificationPipeline import GraphClassificationPipeline

import torch
import numpy as np
import logging


transform = None
# transform = AddRandomEdges(min_density=0.4)
transform = ReduceDiameter(max_diam=26)
# transform = TwoHop()

# batch = datasets.TUDataset(root=f'./../res/datasets/proteins', name='PROTEINS', transform=transform, use_node_attr=True, use_edge_attr=True)
# batch = datasets.TUDataset(root=f'./../res/datasets/mutag', name='MUTAG', transform=transform, use_node_attr=True, use_edge_attr=True)
batch = datasets.TUDataset(root=f'./../res/datasets/enzymes', name='ENZYMES', transform=transform, use_node_attr=True, use_edge_attr=True)
# batch = datasets.TUDataset(root=f'./../res/datasets/collab', name='COLLAB', transform=transform, use_node_attr=True, use_edge_attr=True)
print(f"Number of graphs: {len(batch)}")
print(batch.name)
val_ratio = 0.2
train_dataset, val_dataset, test_dataset = TrainValTestSplitter.graph_class_split(batch, val_size=val_ratio, test_size=1)
val_count = len(val_dataset)

batch_size = 64

train_dloader: DataLoader = DataLoader(train_dataset, 
                                       shuffle=True, 
                                       batch_size=batch_size)

val_dloader: DataLoader = DataLoader(val_dataset, 
                                      shuffle=False, 
                                      batch_size=batch_size)

print("Creating model...")
model = GCN(num_classes=batch.num_classes, 
            num_features=batch.num_features, 
            hidden_size=16)
model.set_classification_task(task='graph')
print("Model created!")

optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)
criterion = torch.nn.NLLLoss()

pipeline = GraphClassificationPipeline(
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
    model=model,
    criterion=criterion,
    optimizer=optimizer
)

_, _, acc = pipeline.train_validate(
    train_data=train_dloader,
    val_data=val_dloader,
    epochs=500,
    log=True
)
np.array(acc).max()

Number of graphs: 600
ENZYMES
Creating model...
Model created!
Epoch: [1/500]         Training loss: 3.5502         Validation loss: 2.4272         Accuracy: 0.1500
Epoch: [2/500]         Training loss: 1.8919         Validation loss: 1.7918         Accuracy: 0.2000
Epoch: [3/500]         Training loss: 1.7630         Validation loss: 1.7523         Accuracy: 0.2500
Epoch: [4/500]         Training loss: 1.7544         Validation loss: 1.7649         Accuracy: 0.1667
Epoch: [5/500]         Training loss: 1.7424         Validation loss: 1.7580         Accuracy: 0.1583
Epoch: [6/500]         Training loss: 1.7403         Validation loss: 1.7503         Accuracy: 0.1833
Epoch: [7/500]         Training loss: 1.7403         Validation loss: 1.7524         Accuracy: 0.1833
Epoch: [8/500]         Training loss: 1.7295         Validation loss: 1.7536         Accuracy: 0.2250
Epoch: [9/500]         Training loss: 1.7298         Validation loss: 1.7555         Accuracy: 0.2167
Epoch: [10/500]    

KeyboardInterrupt: 

In [ ]:
import logging
from torch_geometric import datasets
from rewiring.TrainValTestSplitter import TrainValTestSplitter

logging.basicConfig(level=logging.DEBUG)

dset = datasets.Planetoid(root=f'./../res/datasets/citeseer', name='CiteSeer')
TrainValTestSplitter.single_graph_split(dset=dset, val_size=0.2, test_size=1)

In [4]:
import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch_geometric.nn.conv import GCNConv
from torch_geometric.loader import DataLoader
from torch_geometric.datasets import LRGBDataset

from rewiring.model.GCN import GCN
from rewiring.transform.AddRandomEdges import AddRandomEdges
from rewiring.transform.ReduceDiameter import ReduceDiameter
from rewiring.pipeline.NodeClassificationPipeline import NodeClassifiactionPipeline
from rewiring.TrainValTestSplitter import TrainValTestSplitter
from graphgen.SparseGraphDataset import SparseGraphDataset

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

batch_size = 64
# transform = ReduceDiameter(max_diam=200)
# transform = AddRandomEdges(min_density=0.25)
transform = None

# Load the dataset
print("Loading and transforming training dataset...")
dset = SparseGraphDataset(root=f'./../res/datasets/sparse_dset', transform=transform)

train_dataset, val_dataset, _ = TrainValTestSplitter.graph_class_split(dset)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader =  DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# Create the model
num_features = dset.num_features
num_classes = 5
hidden_size = 64
model = GCN(num_classes=num_classes,
            num_features=num_features,
            hidden_size=hidden_size).to(device)
model.set_classification_task('node')

# Set the loss function and optimizer
criterion = nn.NLLLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

pipeline = NodeClassifiactionPipeline(
    device = device,
    model=model,
    criterion=criterion,
    optimizer=optimizer
)

num_epochs = 10
print("Started training...")
pipeline.train_validate(
    train_data=train_loader, 
    val_data=val_loader, 
    epochs=num_epochs, 
    log=True
)
print("Training finished!")

Loading and transforming training dataset...
This function shouldn't be called, the datasets should 
            have been downloaded to their correct folder during the cloning process!e
            Repository link: https://git01lab.cs.univie.ac.at/peteri00/rewiring-heuristics.git
Started training...


Processing...
Done!


AttributeError: 'list' object has no attribute 'size'

In [4]:
from torch_geometric import datasets
from torch_geometric.transforms import GDC
from rewiring.model.GCN import GCN
from rewiring.model.GIN import GIN

from rewiring.pipeline.SingleGraphNodeClassificationPipeline import SingleGraphNodeClassificationPipeline

from rewiring.transform.AddRandomEdges import AddRandomEdges


import torch
import numpy as np
import logging

# logging.basicConfig(level=logging.INFO)

# transform = AddRandomEdges(min_density=0.2)
# transform = None
transform = GDC()
dset = datasets.Planetoid(root=f'./../res/datasets/citeseer', name='CiteSeer', transform=transform)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GIN(num_classes=dset.num_classes, 
            num_features=dset.num_features)

optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)
criterion = torch.nn.NLLLoss()

pipeline = SingleGraphNodeClassificationPipeline(
    device=device,
    model=model,
    criterion=criterion,
    optimizer=optimizer
)

pipeline.set_masks(data=dset)

_, _, acc = pipeline.train_validate(
    data=dset[0],
    epochs=200,
    log=True
)
np.array(acc).max()

IndexError: index 2587 is out of bounds for dimension 0 with size 1